# Xception + DCGAN: Journal-Ready Implementation with Complete Analysis

## Features:
1. **Fold-Wise GAN Training** (No data leakage)
2. **Ablation Study** (No Aug, Conventional, GAN)
3. **Quantitative GAN Metrics** (FID, SSIM per class & per fold)
4. **Statistical Validation** (Mean±Std, t-test, Wilcoxon, ANOVA)
5. **XAI** (Grad-CAM, Grad-CAM++, Score-CAM, SHAP, LIME)
6. **Per-Class Metrics** (Sensitivity, Specificity, AUC)
7. **ROC/PR Curves**
8. **Learning Curves**
9. **t-SNE Feature Visualization**
10. **Failure Case Analysis**
11. **Training Time Comparison**
12. **Model Calibration** (ECE, Reliability Diagrams)

In [1]:
# ============================
# Imports + Config
# ============================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings
import time
from collections import defaultdict
warnings.filterwarnings('ignore')

from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import Xception, InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, jaccard_score, classification_report, 
                             confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, 
                             average_precision_score, f1_score, precision_score, recall_score)
from sklearn.manifold import TSNE
from scipy import stats
from scipy.linalg import sqrtm
import shap
from lime import lime_image
from skimage.segmentation import slic
import tensorflow as tf

# Config
DATASET_PATH = "/kaggle/input/pmram-bangladeshi-brain-cancer-mri-dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/PMRAM Bangladeshi Brain Cancer - MRI Dataset/Raw Data/Raw"
IMG_SIZE_GAN = (128, 128)
IMG_SIZE_XCEP = (299, 299)
LATENT_DIM = 100
BATCH_SIZE = 32
EPOCHS_GAN = 2000
N_GEN_PER_CLASS = 500
EPOCHS_FE = 10
EPOCHS_FT = 20
N_SPLITS = 5
SEED = 42
NUM_XAI_SAMPLES = 3
SHAP_BG = 20
LIME_SAMPLES = 200

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuration loaded successfully")

2026-04-01 08:33:46.459523: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775032426.682210      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775032426.747305      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775032427.240537      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775032427.240586      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775032427.240589      17 computation_placer.cc:177] computation placer alr

Configuration loaded successfully


## Section 1: GAN Architecture

In [2]:
def build_generator(latent_dim=LATENT_DIM):
    model = Sequential([
        layers.Dense(16*16*256, input_dim=latent_dim),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Reshape((16, 16, 256)),
        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(), layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(3, (5, 5), strides=(1, 1), padding='same', activation='tanh')
    ])
    return model

def build_discriminator(img_shape=(128, 128, 3)):
    model = Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=img_shape),
        layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2), layers.Dropout(0.3),
        layers.Flatten(), layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy')
    return model

def build_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential([generator, discriminator])
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy')
    return model

print("GAN models defined")

GAN models defined


## Section 2: FID and SSIM Metrics

In [3]:
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

def calculate_ssim(real_images, synthetic_images):
    ssim_scores = []
    n_samples = min(len(real_images), len(synthetic_images), 100)
    for i in range(n_samples):
        score = ssim(real_images[i], synthetic_images[i], channel_axis=-1, data_range=1.0)
        ssim_scores.append(score)
    return np.mean(ssim_scores), np.std(ssim_scores)

def calculate_fid(real_images, synthetic_images, batch_size=32):
    inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
    def get_features(images):
        resized = tf.image.resize(images, (299, 299))
        resized = (resized + 1) * 127.5
        preprocessed = inception_preprocess(resized)
        features = inception_model.predict(preprocessed, batch_size=batch_size, verbose=0)
        return features
    real_features = get_features(real_images[:min(len(real_images), 500)])
    synth_features = get_features(synthetic_images[:min(len(synthetic_images), 500)])
    mu_real, sigma_real = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu_synth, sigma_synth = np.mean(synth_features, axis=0), np.cov(synth_features, rowvar=False)
    diff = mu_real - mu_synth
    covmean = sqrtm(sigma_real.dot(sigma_synth))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_synth - 2 * covmean)
    return fid

print("FID and SSIM metrics defined")

FID and SSIM metrics defined


In [4]:
def calculate_per_class_metrics(y_true, y_pred, y_pred_prob, classes):
    cm = confusion_matrix(y_true, y_pred)
    metrics = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1_val = 2 * (precision_val * sensitivity) / (precision_val + sensitivity) if (precision_val + sensitivity) > 0 else 0
        y_true_bin = (y_true == i).astype(int)
        try:
            auc_val = roc_auc_score(y_true_bin, y_pred_prob[:, i])
        except:
            auc_val = 0
        metrics[cls] = {'sensitivity': sensitivity, 'specificity': specificity, 
                       'precision': precision_val, 'f1': f1_val, 'auc': auc_val}
    return metrics

def print_per_class_metrics(metrics, title="Per-Class Metrics"):
    print(f"
{'='*70}")
    print(f"{title}")
    print(f"{'='*70}")
    df = pd.DataFrame(metrics).T
    print(df.round(4).to_string())
    return df

print("Per-class metrics functions defined")

SyntaxError: unterminated f-string literal (detected at line 23) (2273723138.py, line 23)

In [ ]:
def expected_calibration_error(y_true, y_pred_prob, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    confidences = np.max(y_pred_prob, axis=1)
    predictions = np.argmax(y_pred_prob, axis=1)
    accuracies = (predictions == y_true)
    ece = 0
    bin_accuracies, bin_confidences, bin_counts = [], [], []
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            accuracy_in_bin = accuracies[in_bin].mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            bin_accuracies.append(accuracy_in_bin)
            bin_confidences.append(avg_confidence_in_bin)
            bin_counts.append(in_bin.sum())
        else:
            bin_accuracies.append(0)
            bin_confidences.append(0)
            bin_counts.append(0)
    return ece, bin_accuracies, bin_confidences, bin_counts

def plot_reliability_diagram(y_true, y_pred_prob, method_name, save_path):
    ece, bin_accuracies, bin_confidences, bin_counts = expected_calibration_error(y_true, y_pred_prob)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax1.plot(bin_confidences, bin_accuracies, 'o-', label=f'{method_name} (ECE={ece:.4f})')
    ax1.set_xlabel('Mean Predicted Confidence')
    ax1.set_ylabel('Fraction of Positives')
    ax1.set_title('Reliability Diagram')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax2.bar(range(len(bin_counts)), bin_counts, alpha=0.7)
    ax2.set_xlabel('Confidence Bin')
    ax2.set_ylabel('Count')
    ax2.set_title('Prediction Confidence Distribution')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    return ece

print("Model calibration functions defined")

In [ ]:
def plot_roc_curves(y_true, y_pred_prob, classes, method_name, save_path):
    plt.figure(figsize=(12, 10))
    for i, cls in enumerate(classes):
        y_true_bin = (y_true == i).astype(int)
        fpr, tpr, _ = roc_curve(y_true_bin, y_pred_prob[:, i])
        auc = roc_auc_score(y_true_bin, y_pred_prob[:, i])
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves - {method_name}')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_precision_recall_curves(y_true, y_pred_prob, classes, method_name, save_path):
    plt.figure(figsize=(12, 10))
    for i, cls in enumerate(classes):
        y_true_bin = (y_true == i).astype(int)
        precision, recall, _ = precision_recall_curve(y_true_bin, y_pred_prob[:, i])
        avg_precision = average_precision_score(y_true_bin, y_pred_prob[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {avg_precision:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curves - {method_name}')
    plt.legend(loc='lower left')
    plt.grid(alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

print("ROC and PR curve functions defined")

In [ ]:
def analyze_failures(model, val_paths, val_labels, classes, method_name, save_path, n_cases=5):
    val_aug = ImageDataGenerator(rescale=1./255)
    val_df = pd.DataFrame({'filename': val_paths, 'class': val_labels})
    val_gen = val_aug.flow_from_dataframe(val_df, x_col='filename', y_col='class',
        target_size=IMG_SIZE_XCEP, class_mode='categorical', batch_size=BATCH_SIZE, shuffle=False, classes=classes)
    preds = model.predict(val_gen, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    true_classes = val_gen.classes
    confidences = np.max(preds, axis=1)
    false_positives = []
    for i in range(len(true_classes)):
        true_cls = classes[true_classes[i]]
        pred_cls = classes[pred_classes[i]]
        conf = confidences[i]
        if pred_classes[i] != true_classes[i]:
            false_positives.append((i, val_paths[i], true_cls, pred_cls, conf, preds[i]))
    fig, axes = plt.subplots(min(n_cases, len(false_positives)), 3, figsize=(15, 5*min(n_cases, len(false_positives))))
    if n_cases == 1:
        axes = axes.reshape(1, -1)
    for idx in range(min(n_cases, len(false_positives))):
        i, path, true_cls, pred_cls, conf, prob_dist = false_positives[idx]
        img = load_img(path, target_size=IMG_SIZE_XCEP)
        axes[idx, 0].imshow(img)
        axes[idx, 0].set_title(f'Image
True: {true_cls}
Pred: {pred_cls}
Conf: {conf:.3f}')
        axes[idx, 0].axis('off')
        axes[idx, 1].bar(classes, prob_dist)
        axes[idx, 1].set_title('Prediction Probabilities')
        axes[idx, 1].tick_params(axis='x', rotation=45)
        axes[idx, 2].text(0.5, 0.5, f'Failure Analysis:
True: {true_cls}
Predicted: {pred_cls}
Confidence: {conf:.3f}',
                          ha='center', va='center', fontsize=10)
        axes[idx, 2].axis('off')
    plt.suptitle(f'Failure Case Analysis - {method_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"
Failure Analysis for {method_name}: {len(false_positives)} misclassifications")
    return false_positives

print("Failure analysis function defined")

In [ ]:
def plot_tsne_features(model, val_paths, val_labels, classes, method_name, save_path):
    feature_layer = model.get_layer('global_average_pooling2d')
    feature_model = Model(inputs=model.input, outputs=feature_layer.output)
    features, labels_list = [], []
    for path, label in zip(val_paths[:200], val_labels[:200]):
        try:
            img = load_img(path, target_size=IMG_SIZE_XCEP)
            img_array = np.expand_dims(img_to_array(img)/255.0, axis=0)
            feat = feature_model.predict(img_array, verbose=0)
            features.append(feat[0])
            labels_list.append(label)
        except:
            continue
    features = np.array(features)
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(features)-1))
    features_2d = tsne.fit_transform(features)
    plt.figure(figsize=(12, 10))
    colors = plt.cm.rainbow(np.linspace(0, 1, len(classes)))
    for i, cls in enumerate(classes):
        mask = np.array([l == cls for l in labels_list])
        plt.scatter(features_2d[mask, 0], features_2d[mask, 1], c=[colors[i]], label=cls, alpha=0.6, s=50)
    plt.legend()
    plt.title(f't-SNE Feature Visualization - {method_name}', fontsize=14, fontweight='bold')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.grid(alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"t-SNE plot saved: {save_path}")

print("t-SNE visualization function defined")

In [ ]:
def plot_learning_curves(history_fe, history_ft, method_name, fold_num, save_path):
    train_loss = history_fe.history['loss'] + history_ft.history['loss']
    val_loss = history_fe.history['val_loss'] + history_ft.history['val_loss']
    train_acc = history_fe.history['accuracy'] + history_ft.history['accuracy']
    val_acc = history_fe.history['val_accuracy'] + history_ft.history['val_accuracy']
    epochs = range(1, len(train_loss) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, train_loss, 'b-', label='Training Loss')
    ax1.plot(epochs, val_loss, 'r-', label='Validation Loss')
    ax1.axvline(x=len(history_fe.history['loss']), color='g', linestyle='--', label='Start Fine-tuning')
    ax1.set_title(f'Loss Curves - {method_name} Fold {fold_num}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax2.plot(epochs, train_acc, 'b-', label='Training Accuracy')
    ax2.plot(epochs, val_acc, 'r-', label='Validation Accuracy')
    ax2.axvline(x=len(history_fe.history['accuracy']), color='g', linestyle='--', label='Start Fine-tuning')
    ax2.set_title(f'Accuracy Curves - {method_name} Fold {fold_num}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

print("Learning curves function defined")